In [1]:
!unzip -q /content/fitzpatrick17k.zip -d /content/data/

unzip:  cannot find or open /content/fitzpatrick17k.zip, /content/fitzpatrick17k.zip.zip or /content/fitzpatrick17k.zip.ZIP.


In [2]:
!unzip -o -q /content/fitzpatrick17k.zip -d /content/

# Verify both exist
import os
print("CSV found:", os.path.exists('/content/fitzpatrick17k.csv'))
print("Image directory found:", os.path.exists('/content/data/finalfitz17k'))

# Optionally, count files in the image directory to ensure content
if os.path.exists('/content/data/finalfitz17k'):
    print("Number of image files:", len(os.listdir('/content/data/finalfitz17k')))
else:
    print("Warning: Image directory /content/data/finalfitz17k not found after extraction.")

unzip:  cannot find or open /content/fitzpatrick17k.zip, /content/fitzpatrick17k.zip.zip or /content/fitzpatrick17k.zip.ZIP.
CSV found: False
Image directory found: False


In [3]:
import os
print("Contents of /content/ directory:")
for item in os.listdir('/content/'):
    print(item)

# Also check inside /content/data/ just in case
if os.path.exists('/content/data/') and os.path.isdir('/content/data/'):
    print("\nContents of /content/data/ directory:")
    for item in os.listdir('/content/data/'):
        print(item)

Contents of /content/ directory:
.config
drive
sample_data


In [4]:
import os

csv  = '/content/fitzpatrick17k.csv'
imgs = '/content/data/finalfitz17k'

print("CSV  :", "✅" if os.path.exists(csv) else "❌ NOT FOUND")
print("Images:", f"✅ {len(os.listdir(imgs))} files" if os.path.exists(imgs) else "❌ NOT FOUND")

CSV  : ❌ NOT FOUND
Images: ❌ NOT FOUND


In [5]:
import os

print("Contents of /content/data/data/ directory:")
if os.path.exists('/content/data/data/') and os.path.isdir('/content/data/data/'):
    for item in os.listdir('/content/data/data/'):
        print(item)
else:
    print("Directory /content/data/data/ does not exist or is not a directory.")

Contents of /content/data/data/ directory:
Directory /content/data/data/ does not exist or is not a directory.


In [6]:
!unzip -l /content/fitzpatrick17k.zip

unzip:  cannot find or open /content/fitzpatrick17k.zip, /content/fitzpatrick17k.zip.zip or /content/fitzpatrick17k.zip.ZIP.


In [7]:
%%writefile /content/train_final.py
import os, sys, argparse
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, classification_report
from torchvision import models, transforms
from PIL import Image

# ── PATHS ──────────────────────────────────────────────────────
CSV_PATH = "/content/fitzpatrick17k.csv"
IMG_DIR  = "/content/data/finalfitz17k"
CKPT_DIR = Path("/content/models/skin_classifier")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CONFIG ─────────────────────────────────────────────────────
SEED            = 42
BATCH_SIZE      = 32
NUM_WORKERS     = 2
IMG_SIZE        = 224
WARMUP_EPOCHS   = 8
FINETUNE_EPOCHS = 25 # Keep this as original total for context, but adjust loop
WARMUP_LR       = 1e-3
FINETUNE_LR     = 5e-5
DROPOUT         = 0.4
WEIGHT_DECAY    = 1e-4
UNFREEZE_BLOCK  = 5
LABEL_MAP       = {"malignant": 0, "benign": 1, "non-neoplastic": 2}
CLASS_NAMES     = ["Malignant", "Benign", "Non-neoplastic"]
NUM_CLASSES     = 3

# ── DATASET ────────────────────────────────────────────────────
class FitzDataset(Dataset):
    def __init__(self, df, img_dir, split="train"):
        self.df      = df.reset_index(drop=True)
        self.img_dir = img_dir
        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]
        if split == "train":
            self.tfm = transforms.Compose([
                transforms.Resize(256),
                transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
                transforms.RandomRotation(15),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])
        else:
            self.tfm = transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(IMG_SIZE),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.img_dir, f"{row['md5hash']}.jpg")
        try:
            img = Image.open(path).convert("RGB")
        except:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128,128,128))
        return self.tfm(img), LABEL_MAP[row["three_partition_label"]]

# ── MODEL ──────────────────────────────────────────────────────
def build_model():
    m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    for p in m.parameters(): p.requires_grad = False
    m.classifier = nn.Sequential(
        nn.Dropout(DROPOUT, inplace=True),
        nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    )
    return m

def unfreeze(model, from_block=5):
    for name, p in model.named_parameters():
        if "classifier" in name: p.requires_grad = True; continue
        try:
            if int(name.split(".")[1]) >= from_block: p.requires_grad = True
        except: pass

# ── TRAIN/EVAL ─────────────────────────────────────────────────
def run_epoch(model, loader, criterion, opt, device, train=True):
    model.train() if train else model.eval()
    loss_sum = correct = total = 0
    preds_all = []; labels_all = []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * imgs.size(0)
            p = logits.argmax(1)
            correct += (p == labels).sum().item()
            total   += imgs.size(0)
            preds_all.extend(p.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    return loss_sum/total, correct/total, np.array(preds_all), np.array(labels_all)

# ── MAIN ───────────────────────────────────────────────────────
import random
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if device.type=='cuda' else 'N/A'}")

# Load + clean CSV
print(f"Debug: CSV_PATH = {CSV_PATH}, exists = {os.path.exists(CSV_PATH)}") # Re-insert debug line
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["md5hash","three_partition_label"])
df["three_partition_label"] = df["three_partition_label"].str.strip().str.lower()
df = df[df["three_partition_label"].isin(LABEL_MAP)]
if "qc" in df.columns:
    df = df[df["qc"].isna() | (df["qc"].str.strip().str.lower() != "bad")]
print(df["three_partition_label"].value_counts().to_string())
print(f"Total valid: {len(df)}")

# Split
train_df, temp = train_test_split(df, test_size=0.2, stratify=df["three_partition_label"], random_state=SEED)
val_df, test_df = train_test_split(temp, test_size=0.5, stratify=temp["three_partition_label"], random_state=SEED)
print(f"Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}")

# Loaders
tl  = DataLoader(FitzDataset(train_df, IMG_DIR, "train"), batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
vl  = DataLoader(FitzDataset(val_df,   IMG_DIR, "val"),   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
tel = DataLoader(FitzDataset(test_df,  IMG_DIR, "test"),  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Class weights
counts = df["three_partition_label"].value_counts()
cw = torch.tensor([len(df)/(NUM_CLASSES*counts[c]) for c in ["malignant","benign","non-neoplastic"]], dtype=torch.float32).to(device)
print(f"Class weights: {cw.cpu().numpy().round(3)}")

model     = build_model().to(device)
criterion = nn.CrossEntropyLoss(weight=cw)
best_ckpt = CKPT_DIR / "best_model.pth"
best_val  = 0.0

# ── Phase 1: Warmup ────────────────────────────────────────────
print(f"\n{'='*55}\nPHASE 1 — Warmup ({WARMUP_EPOCHS} epochs, head only)\n{'='*55}")
opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=WARMUP_LR, weight_decay=WEIGHT_DECAY)
sch = CosineAnnealingLR(opt, T_max=WARMUP_EPOCHS, eta_min=1e-5)
for ep in range(1, WARMUP_EPOCHS+1):
    tl_, ta, _, _ = run_epoch(model, tl, criterion, opt, device, train=True)
    vl_, va, _, _ = run_epoch(model, vl, criterion, opt, device, train=False)
    sch.step()
    flag = ""
    if va > best_val: best_val=va; torch.save(model.state_dict(), best_ckpt); flag=" 💾"
    print(f"E{ep:02}/{WARMUP_EPOCHS} | train {ta:.4f} | val {va:.4f} | val_loss {vl_:.4f}{flag}")

# ── Phase 2: Fine-tune ─────────────────────────────────────────
print(f"\n{'='*55}\nPHASE 2 — Fine-tune (Continuing for epochs 24 and 25)\n{'='*55}")
# Load the best model found so far to resume fine-tuning
if best_ckpt.exists():
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    print(f"Loaded best model from {best_ckpt} to continue fine-tuning.")
unfreeze(model, UNFREEZE_BLOCK)
opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY)
# Re-initialize scheduler for the remaining 2 epochs
sch = CosineAnnealingLR(opt, T_max=2, eta_min=1e-6) # T_max for the *remaining* epochs

# Loop only for the remaining epochs (24 and 25)
for ep_relative in range(1, 3): # Running 2 epochs
    current_global_epoch = 23 + ep_relative # Calculate global epoch number for printing
    tl_, ta, _, _ = run_epoch(model, tl, criterion, opt, device, train=True)
    vl_, va, _, _ = run_epoch(model, vl, criterion, opt, device, train=False)
    sch.step()
    flag = ""
    if va > best_val: best_val=va; torch.save(model.state_dict(), best_ckpt); flag=" 💾"
    print(f"E{current_global_epoch:02}/{FINETUNE_EPOCHS} | train {ta:.4f} | val {va:.4f} | val_loss {vl_:.4f}{flag}")

# ── Test ───────────────────────────────────────────────────────
print(f"\n{'='*55}\nFINAL TEST\n{'='*55}")
model.load_state_dict(torch.load(best_ckpt, map_location=device))
_, acc, preds, labels = run_epoch(model, tel, criterion, None, device, train=False)
kappa = cohen_kappa_score(labels, preds)
print(f"Test Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"Cohen's Kappa : {kappa:.4f}")
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

# ── ONNX ───────────────────────────────────────────────────────
onnx_path = str(CKPT_DIR / "skin_classifier_b3.onnx")
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
model.eval()
torch.onnx.export(model, dummy, onnx_path,
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image":{0:"batch"},"logits":{0:"batch"}},
    opset_version=18)
print(f"\n✅ ONNX  → {onnx_path}")
print(f"✅ Best  → {best_ckpt}")

Writing /content/train_final.py


In [11]:
!python /content/train_final.py

Device: cuda | GPU: Tesla T4
Debug: CSV_PATH = /content/fitzpatrick17k.csv, exists = True
three_partition_label
non-neoplastic    12080
malignant          2263
benign             2234
Total valid: 16577
Train:13261 Val:1658 Test:1658
Class weights: [2.442 2.473 0.457]
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100% 47.2M/47.2M [00:00<00:00, 172MB/s]

PHASE 1 — Warmup (8 epochs, head only)
E01/8 | train 0.3626 | val 0.1369 | val_loss 1.1012 💾
E02/8 | train 0.3564 | val 0.7286 | val_loss 1.1450 💾
E03/8 | train 0.3507 | val 0.1369 | val_loss 1.1403
E04/8 | train 0.3669 | val 0.7286 | val_loss 1.1259
E05/8 | train 0.3677 | val 0.7286 | val_loss 1.1114
E06/8 | train 0.3649 | val 0.7286 | val_loss 1.1113
E07/8 | train 0.3658 | val 0.7286 | val_loss 1.1007
E08/8 | train 0.3612 | val 0.7286 | val_loss 1.1022

PHASE 2 — Fine-tune (Continuing for epochs 24 and 25)
Loaded be

In [12]:
%%writefile /content/train_robust.py
import os, sys, random
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, classification_report, f1_score, confusion_matrix
from torchvision import models, transforms
from PIL import Image

# ── PATHS ──────────────────────────────────────────────────────
CSV_PATH = "/content/data/fitzpatrick17k.csv"
IMG_DIR  = "/content/data/finalfitz17k"
CKPT_DIR = Path("/content/models/skin_classifier")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CONFIG ─────────────────────────────────────────────────────
SEED            = 42
BATCH_SIZE      = 32
NUM_WORKERS     = 2
IMG_SIZE        = 224
WARMUP_EPOCHS   = 8
FINETUNE_EPOCHS = 25
WARMUP_LR       = 1e-4      # ← lowered from 1e-3, was too aggressive for new head
FINETUNE_LR     = 1e-5      # ← lowered from 5e-5
DROPOUT         = 0.4
WEIGHT_DECAY    = 1e-4
UNFREEZE_BLOCK  = 5
GRAD_CLIP       = 1.0       # ← gradient clipping prevents collapse
LABEL_SMOOTH    = 0.1       # ← softens overconfident predictions
PATIENCE        = 6         # ← early stop if no macro-F1 improvement
LABEL_MAP       = {"malignant": 0, "benign": 1, "non-neoplastic": 2}
CLASS_NAMES     = ["Malignant", "Benign", "Non-neoplastic"]
NUM_CLASSES     = 3

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── DATASET ────────────────────────────────────────────────────
class FitzDataset(Dataset):
    def __init__(self, df, img_dir, split="train"):
        self.df      = df.reset_index(drop=True)
        self.img_dir = img_dir
        mean = [0.485, 0.456, 0.406]; std = [0.229, 0.224, 0.225]
        if split == "train":
            self.tfm = transforms.Compose([
                transforms.Resize(256),
                transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
                transforms.RandomRotation(15),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])
        else:
            self.tfm = transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(IMG_SIZE),
                transforms.ToTensor(),
                transforms.Normalize(mean, std),
            ])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.img_dir, f"{row['md5hash']}.jpg")
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        return self.tfm(img), LABEL_MAP[row["three_partition_label"]]

# ── MODEL ──────────────────────────────────────────────────────
def build_model():
    m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    for p in m.parameters(): p.requires_grad = False
    m.classifier = nn.Sequential(
        nn.Dropout(DROPOUT, inplace=True),
        nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    )
    return m

def unfreeze(model, from_block=5):
    for name, p in model.named_parameters():
        if "classifier" in name: p.requires_grad = True; continue
        try:
            if int(name.split(".")[1]) >= from_block: p.requires_grad = True
        except Exception: pass

# ── TRAIN/EVAL ─────────────────────────────────────────────────
def run_epoch(model, loader, criterion, opt, device, train=True):
    model.train() if train else model.eval()
    loss_sum = correct = total = 0
    preds_all, labels_all = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if train:
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
            loss_sum += loss.item() * imgs.size(0)
            p = logits.argmax(1)
            correct += (p == labels).sum().item()
            total   += imgs.size(0)
            preds_all.extend(p.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    macro_f1 = f1_score(labels_all, preds_all, average="macro", zero_division=0)
    return loss_sum/total, correct/total, macro_f1, np.array(preds_all), np.array(labels_all)

# ── MAIN ───────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if device.type=='cuda' else 'N/A'}")

df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["md5hash", "three_partition_label"])
df["three_partition_label"] = df["three_partition_label"].str.strip().str.lower()
df = df[df["three_partition_label"].isin(LABEL_MAP)]
if "qc" in df.columns:
    df = df[df["qc"].isna() | (df["qc"].str.strip().str.lower() != "bad")]
print(df["three_partition_label"].value_counts().to_string())
print(f"Total valid: {len(df)}")

train_df, temp = train_test_split(df, test_size=0.2, stratify=df["three_partition_label"], random_state=SEED)
val_df, test_df = train_test_split(temp, test_size=0.5, stratify=temp["three_partition_label"], random_state=SEED)
print(f"Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}")

train_ds = FitzDataset(train_df, IMG_DIR, "train")
val_ds   = FitzDataset(val_df,   IMG_DIR, "val")
test_ds  = FitzDataset(test_df,  IMG_DIR, "test")

# ── Balanced sampler (replaces aggressive loss-weighting) ───────
counts = train_df["three_partition_label"].value_counts()
sample_w = train_df["three_partition_label"].map(lambda c: 1.0 / counts[c]).values
sampler = WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)

tl  = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
vl  = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=NUM_WORKERS, pin_memory=True)
tel = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=NUM_WORKERS, pin_memory=True)

model     = build_model().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)   # mild smoothing, NO heavy class weights
best_ckpt = CKPT_DIR / "best_model.pth"
best_macro_f1 = 0.0
patience_ctr  = 0

def maybe_save(macro_f1, val_acc, preds, labels, ep, phase):
    global best_macro_f1, patience_ctr
    flag = ""
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save(model.state_dict(), best_ckpt)
        flag = " 💾"
        patience_ctr = 0
    else:
        patience_ctr += 1
    cm = confusion_matrix(labels, preds)
    print(f"{phase} E{ep:02} | val_acc {val_acc:.4f} | macro_F1 {macro_f1:.4f}{flag} | "
          f"preds_per_class {np.bincount(preds, minlength=3)}")
    if patience_ctr == 1 or ep % 5 == 0:
        print(f"   confusion matrix:\n{cm}")
    return patience_ctr

# ── Phase 1: Warmup ──────────────────────────────────────────────
print(f"\n{'='*55}\nPHASE 1 — Warmup ({WARMUP_EPOCHS} epochs, head only, lr={WARMUP_LR})\n{'='*55}")
opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=WARMUP_LR, weight_decay=WEIGHT_DECAY)
sch = CosineAnnealingLR(opt, T_max=WARMUP_EPOCHS, eta_min=1e-6)
for ep in range(1, WARMUP_EPOCHS + 1):
    tl_, ta, tf1, _, _ = run_epoch(model, tl, criterion, opt, device, train=True)
    vl_, va, vf1, vp, vlab = run_epoch(model, vl, criterion, opt, device, train=False)
    sch.step()
    maybe_save(vf1, va, vp, vlab, ep, "Warmup")

# ── Phase 2: Fine-tune ───────────────────────────────────────────
print(f"\n{'='*55}\nPHASE 2 — Fine-tune ({FINETUNE_EPOCHS} epochs, blocks {UNFREEZE_BLOCK}+, lr={FINETUNE_LR})\n{'='*55}")
unfreeze(model, UNFREEZE_BLOCK)
opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY)
sch = CosineAnnealingLR(opt, T_max=FINETUNE_EPOCHS, eta_min=1e-7)
patience_ctr = 0
for ep in range(1, FINETUNE_EPOCHS + 1):
    tl_, ta, tf1, _, _ = run_epoch(model, tl, criterion, opt, device, train=True)
    vl_, va, vf1, vp, vlab = run_epoch(model, vl, criterion, opt, device, train=False)
    sch.step()
    pc = maybe_save(vf1, va, vp, vlab, ep, "Finetune")
    if pc >= PATIENCE:
        print(f"   ⏹ Early stopping — no macro-F1 improvement for {PATIENCE} epochs")
        break

# ── Test ───────────────────────────────────────────────────────
print(f"\n{'='*55}\nFINAL TEST\n{'='*55}")
model.load_state_dict(torch.load(best_ckpt, map_location=device))
_, acc, macro_f1, preds, labels = run_epoch(model, tel, criterion, None, device, train=False)
kappa = cohen_kappa_score(labels, preds)
print(f"Test Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Cohen's Kappa : {kappa:.4f}")
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(labels, preds))

# ── ONNX export (legacy exporter — avoids onnxscript dependency) ─
onnx_path = str(CKPT_DIR / "skin_classifier_b3.onnx")
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
model.eval()
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13,
    dynamo=False,          # ← forces legacy exporter, no onnxscript needed
)
print(f"\n✅ ONNX  → {onnx_path}")
print(f"✅ Best  → {best_ckpt}")


Writing /content/train_robust.py


In [15]:
import os
import shutil

zip_path = '/content/fitzpatrick17k.zip'
csv_expected_path = '/content/data/fitzpatrick17k.csv'
img_dir_expected_path = '/content/data/finalfitz17k'
data_target_dir = '/content/data/'
csv_old_path = '/content/fitzpatrick17k.csv'

print(f"Checking for zip file: {zip_path}")
if os.path.exists(zip_path):
    print("Zip file found.")

    # Clean up any existing /content/data directory
    if os.path.exists(data_target_dir):
        print(f"Removing existing directory: {data_target_dir}")
        shutil.rmtree(data_target_dir)

    # Create the target directory for unzipping
    os.makedirs(data_target_dir, exist_ok=True)
    print(f"Created directory: {data_target_dir}")

    print("Unzipping... This may take a moment.")
    # Unzip the file into /content/data/
    !unzip -o -q {zip_path} -d {data_target_dir}
    print("Unzip complete.")

    # Verify both exist in the new location
    print("\nVerifying extracted files:")
    print(f"CSV found at {csv_expected_path}:", os.path.exists(csv_expected_path))
    print(f"Image directory found at {img_dir_expected_path}:", os.path.exists(img_dir_expected_path))

    if os.path.exists(img_dir_expected_path):
        print("Number of image files:", len(os.listdir(img_dir_expected_path)))
    else:
        print(f"Warning: Image directory {img_dir_expected_path} not found after extraction.")

    # Remove the CSV that might be directly in /content/
    if os.path.exists(csv_old_path) and csv_old_path != csv_expected_path:
        print(f"Removing old CSV file from {csv_old_path}")
        os.remove(csv_old_path)

    if os.path.exists(csv_expected_path) and os.path.exists(img_dir_expected_path):
        print("\n✅ Data successfully extracted and verified to match script paths. You can now re-run the training script (`!python /content/train_robust.py`).")
    else:
        print("\n❌ Error: Required files/directories not found after extraction. Please check the zip file content or path.")
else:
    print(f"❌ Error: Zip file not found at {zip_path}. Please ensure it is uploaded.")

Checking for zip file: /content/fitzpatrick17k.zip
Zip file found.
Removing existing directory: /content/data/
Created directory: /content/data/
Unzipping... This may take a moment.
Unzip complete.

Verifying extracted files:
CSV found at /content/data/fitzpatrick17k.csv: False
Image directory found at /content/data/finalfitz17k: False

❌ Error: Required files/directories not found after extraction. Please check the zip file content or path.


In [16]:
!unzip -l /content/fitzpatrick17k.zip

Streaming output truncated to the last 5000 lines.
    45609  2021-05-07 20:45   data/finalfitz17k/3b843066b041e22f335a890eae042b2a.jpg
    45638  2021-05-07 20:43   data/finalfitz17k/ad9d434175dfebece8811512b789af06.jpg
    33159  2021-05-07 20:30   data/finalfitz17k/9313cd83e63de37e82a3b69b31776b45.jpg
    54827  2021-05-07 20:31   data/finalfitz17k/374d3ae5b810686437778c5df9133448.jpg
    41412  2021-05-07 20:39   data/finalfitz17k/08c31b995b51e21dcb10fb8d900bbaa2.jpg
   119231  2021-05-07 20:39   data/finalfitz17k/05c8b20cc66dec4c74295e97935edcc6.jpg
   197029  2021-05-07 20:39   data/finalfitz17k/1b8e1b61ab30fd12129793b30791d247.jpg
   198120  2021-05-07 20:34   data/finalfitz17k/ba0322417bc6fcf4280d46182c1c18a0.jpg
    85376  2021-05-07 20:32   data/finalfitz17k/f71b5dee1c5ed4b2ff34f8ceee548b3f.jpg
    55909  2021-05-07 20:47   data/finalfitz17k/28a5842f0d95823b74bf98ce62e2a23f.jpg
    38098  2021-05-07 20:41   data/finalfitz17k/f9c6638164ca8f75b864bc17164ce9b6.jpg
    56750  202

In [22]:
import shutil, os

# Clean up the broken double-nested folder
if os.path.exists('/content/data'):
    shutil.rmtree('/content/data')

# Unzip to /content/ directly — NOT /content/data/
# (the zip's internal structure already has data/finalfitz17k/...)
!unzip -o -q /content/fitzpatrick17k.zip -d /content/

# Verify
img_dir = '/content/data/finalfitz17k'
csv_path = '/content/fitzpatrick17k.csv'   # this is uploaded separately, already correct

print("Images found:", len(os.listdir(img_dir)) if os.path.exists(img_dir) else "❌ NOT FOUND")
print("CSV exists:", os.path.exists(csv_path))

Images found: 16577
CSV exists: True


In [20]:
CSV_PATH = "/content/fitzpatrick17k.csv"      # ← stays at root, was uploaded separately
IMG_DIR  = "/content/data/finalfitz17k"       # ← one level under data/, NOT double-nested

In [24]:
!python /content/train_robust.py

Device: cuda | GPU: Tesla T4
three_partition_label
non-neoplastic    12080
malignant          2263
benign             2234
Total valid: 16577
Train:13261 Val:1658 Test:1658

PHASE 1 — Warmup (8 epochs, head only, lr=0.0001)
Warmup E01 | val_acc 0.4916 | macro_F1 0.4171 💾 | preds_per_class [632 351 675]
Warmup E02 | val_acc 0.5157 | macro_F1 0.4343 💾 | preds_per_class [523 406 729]
Warmup E03 | val_acc 0.5127 | macro_F1 0.4393 💾 | preds_per_class [536 417 705]
Warmup E04 | val_acc 0.5163 | macro_F1 0.4362 | preds_per_class [512 420 726]
   confusion matrix:
[[144  38  45]
 [ 78  88  57]
 [290 294 624]]
Warmup E05 | val_acc 0.5676 | macro_F1 0.4660 💾 | preds_per_class [436 382 840]
   confusion matrix:
[[132  40  55]
 [ 67  90  66]
 [237 252 719]]
Warmup E06 | val_acc 0.5157 | macro_F1 0.4376 | preds_per_class [542 404 712]
   confusion matrix:
[[151  36  40]
 [ 81  87  55]
 [310 281 617]]
Warmup E07 | val_acc 0.5446 | macro_F1 0.4530 | preds_per_class [472 396 790]
Warmup E08 | val_acc 

In [25]:
!pip install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 20.8 MB/s eta 0:00:00


In [26]:
import torch
import sys
from pathlib import Path

# Add your script folder to path so it can find your model.py
sys.path.insert(0, '/content')
from model import build_model

# Setup paths and parameters
CKPT_PATH = "/content/models/skin_classifier/best_model.pth"
ONNX_PATH = "/content/models/skin_classifier/skin_classifier_b3.onnx"
NUM_CLASSES = 3
IMG_SIZE = 224

# Load the saved model
print("Loading saved weights...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(num_classes=NUM_CLASSES, dropout=0.4)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.to(device)
model.eval()

# Export to ONNX
print("Exporting to ONNX...")
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={"image": {0: "batch_size"}, "logits": {0: "batch_size"}},
    opset_version=18
)

print(f"✅ ONNX successfully exported to: {ONNX_PATH}")

Loading saved weights...
Exporting to ONNX...


/tmp/ipykernel_547/2762516264.py:26: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ ONNX successfully exported to: /content/models/skin_classifier/skin_classifier_b3.onnx
